In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

True

Step 1: Generic Schema

In [36]:
from pydantic import BaseModel
from typing import Optional, List, Union


class LabTest(BaseModel):
    name: str
    value: Optional[Union[float, str]] = None
    unit: Optional[str] = None
    reference_range: Optional[str] = None


class BloodReport(BaseModel):
    patient_name: Optional[str] = None
    age: Optional[int] = None
    gender: Optional[str] = None
    report_date: Optional[str] = None

    tests: List[LabTest]

In [37]:
with open("report2.txt", "r", encoding="utf-8") as f:
    report_text = f.read()

In [38]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

extractor = llm.with_structured_output(BloodReport)

In [39]:
EXTRACTION_PROMPT = """
You are a medical report extraction system.

Extract all patient information and laboratory tests from the report.

Rules:

1. Extract patient name if available.
2. Extract age if available.
3. Extract gender if available.
4. Extract report date if available.

For every laboratory test found:

- test name
- measured value
- unit
- reference range

Do not analyze the report.
Do not provide medical advice.
Do not determine whether values are high or low.

Only extract factual information present in the report.

If a field is missing, return null.

Report:

{report}
"""

In [40]:
result = extractor.invoke(
    EXTRACTION_PROMPT.format(
        report=report_text
    )
)

In [41]:
print(result.model_dump())

{'patient_name': 'John Doe', 'age': 38, 'gender': 'Male', 'report_date': '06 June 2026', 'tests': [{'name': 'Hemoglobin', 'value': 14.8, 'unit': 'g/dL', 'reference_range': '13.5–17.5 g/dL'}, {'name': 'WBC Count', 'value': 6.7, 'unit': '×10³/µL', 'reference_range': '4.0–11.0 ×10³/µL'}, {'name': 'Platelet Count', 'value': 255.0, 'unit': '×10³/µL', 'reference_range': '150–450 ×10³/µL'}, {'name': 'Glucose (Fasting)', 'value': 92.0, 'unit': 'mg/dL', 'reference_range': '70–99 mg/dL'}, {'name': 'Creatinine', 'value': 0.9, 'unit': 'mg/dL', 'reference_range': '0.7–1.3 mg/dL'}, {'name': 'Sodium', 'value': 140.0, 'unit': 'mmol/L', 'reference_range': '135–145 mmol/L'}, {'name': 'Potassium', 'value': 4.2, 'unit': 'mmol/L', 'reference_range': '3.5–5.0 mmol/L'}, {'name': 'Total Cholesterol', 'value': 182.0, 'unit': 'mg/dL', 'reference_range': '<200 mg/dL'}, {'name': 'HDL Cholesterol', 'value': 56.0, 'unit': 'mg/dL', 'reference_range': '>40 mg/dL'}, {'name': 'LDL Cholesterol', 'value': 102.0, 'unit': 

In [42]:
from pydantic import BaseModel
from typing import List


class HealthAnalysis(BaseModel):
    overall_summary: str
    concerns: List[str]
    diet_recommendations: List[str]
    exercise_recommendations: List[str]

In [43]:
advisor = llm.with_structured_output(
    HealthAnalysis
)

In [44]:
analysis = advisor.invoke(
    f"""
    Patient Information:

    {result.model_dump_json(indent=2)}

    Generate:

    1. Overall health summary
    2. Main concerns
    3. Diet recommendations
    4. Exercise recommendations

    Do not diagnose diseases.
    """
)

In [45]:
analysis

HealthAnalysis(overall_summary="John Doe's test results indicate that he is generally in good health, with most of his test values falling within the reference ranges. However, his total cholesterol and LDL cholesterol levels are near the upper limits of the reference ranges, which may be a concern. His triglycerides level is also near the upper limit of the reference range.", concerns=['Total Cholesterol near upper limit', 'LDL Cholesterol near upper limit', 'Triglycerides near upper limit'], diet_recommendations=['Maintain a balanced diet low in saturated and trans fats', 'Increase consumption of fruits, vegetables, and whole grains', 'Choose lean protein sources and low-fat dairy products'], exercise_recommendations=['Engage in regular aerobic exercise, such as brisk walking, cycling, or swimming, for at least 150 minutes per week', 'Incorporate strength-training exercises into your routine, focusing on major muscle groups, at least two times per week'])